# Params

In [46]:
DATASET_DIR = './data'

TRAIN_MODEL = True
MODEL_SAVE_PATH = "./ner_model"

LR = 2e-5
BATCH_SIZE = 16
NUM_EPOCHS = 4
WEIGHT_DECAY = 0.01

# Setup

In [47]:
!pip install seqeval

In [48]:
import numpy as np
import pandas as pd
import kagglehub
import os
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments
import torch

import warnings
warnings.filterwarnings(
    "ignore",
    message="Was asked to gather along dimension 0"
)
warnings.filterwarnings(
    "ignore",
    message="There were missing keys in the checkpoint model loaded"
)

print(torch.__version__)

2.10.0+cu128


In [49]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0)) 

True
Tesla T4


# Dataset handling
Loading, parsing, mapping, tokenizing

In [50]:
if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)
    
path = kagglehub.dataset_download("alaakhaled/conll003-englishversion", output_dir=DATASET_DIR)
print(path)
print(os.listdir(path))

/kaggle/input/datasets/alaakhaled/conll003-englishversion
['valid.txt', 'metadata', 'test.txt', 'train.txt']


In [51]:
def load_conll_ner(file_path):
    sentences = []
    tokens, labels = [], []
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            if not line:
                if tokens:
                    sentences.append({"tokens": tokens, "ner_tags": labels})
                    tokens, labels = [], []
                continue
            
            if line.startswith("-DOCSTART-"):
                continue
            
            
            word, pos, chunk, ner = line.split()
            tokens.append(word)
            labels.append(ner)
    if tokens:
        sentences.append({"tokens": tokens, "ner_tags": labels})

    return sentences


train_data = load_conll_ner(f"{path}/train.txt")
val_data = load_conll_ner(f"{path}/valid.txt")
test_data  = load_conll_ner(f"{path}/test.txt")

train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)
test_dataset  = Dataset.from_list(test_data)

print(f"Number of training sentences: {len(train_dataset)}")
print(f"Number of validation sentences: {len(val_dataset)}")
print(f"Number of test sentences: {len(test_dataset)}")

print("Example sentence:")
print(train_dataset[0])

Number of training sentences: 14041
Number of validation sentences: 3250
Number of test sentences: 3453
Example sentence:
{'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'ner_tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']}


In [52]:
# create label mapping
label_list = sorted(set(label for ex in train_dataset for label in ex["ner_tags"]))
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}
print("Label to ID mapping:")
print(label2id)
print("ID to Label mapping:")
print(id2label)


Label to ID mapping:
{'B-LOC': 0, 'B-MISC': 1, 'B-ORG': 2, 'B-PER': 3, 'I-LOC': 4, 'I-MISC': 5, 'I-ORG': 6, 'I-PER': 7, 'O': 8}
ID to Label mapping:
{0: 'B-LOC', 1: 'B-MISC', 2: 'B-ORG', 3: 'B-PER', 4: 'I-LOC', 5: 'I-MISC', 6: 'I-ORG', 7: 'I-PER', 8: 'O'}


In [53]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_dataset(dataset):
    def tokenize_and_align_labels(example):
        tokenized = tokenizer(
            example["tokens"],
            truncation=True,
            is_split_into_words=True
        )

        word_ids = tokenized.word_ids()
        labels = []
        previous_word = None
    
        for word_id in word_ids:
            if word_id is None:
                labels.append(-100)  # ignore
            elif word_id != previous_word:
                labels.append(label2id[example["ner_tags"][word_id]])
            else:
                labels.append(-100)  # ignore subword tokens
            previous_word = word_id
    
        tokenized["labels"] = labels
        return tokenized
        
    dataset = dataset.map(tokenize_and_align_labels, batched=False)
    dataset = dataset.remove_columns(["tokens", "ner_tags"])
    return dataset

train_dataset = tokenize_dataset(train_dataset)
val_dataset = tokenize_dataset(val_dataset)
test_dataset = tokenize_dataset(test_dataset)

print("Tokenized example:")
print(train_dataset[0])

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Tokenized example:
{'input_ids': [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 2, 8, 1, 8, 8, 8, 1, 8, -100, 8, -100]}


In [54]:
data_collator = DataCollatorForTokenClassification(tokenizer)

batch = data_collator([
    train_dataset[0],
    train_dataset[1]
])
print(batch)

{'input_ids': tensor([[  101,  7270, 22961,  1528,  1840,  1106, 21423,  1418,  2495, 12913,
           119,   102],
        [  101,  1943, 14428,   102,     0,     0,     0,     0,     0,     0,
             0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]]), 'labels': tensor([[-100,    2,    8,    1,    8,    8,    8,    1,    8, -100,    8, -100],
        [-100,    3,    7, -100, -100, -100, -100, -100, -100, -100, -100, -100]])}


# BERT
creating, training

In [55]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir=MODEL_SAVE_PATH,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [56]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        cur_preds = []
        cur_labels = []

        for p_i, l_i in zip(pred, lab):
            if l_i != -100:
                cur_preds.append(id2label[p_i])
                cur_labels.append(id2label[l_i])

        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

if TRAIN_MODEL:
    trainer.train()
    trainer.save_model(MODEL_SAVE_PATH + "/final_model")
    tokenizer.save_pretrained(MODEL_SAVE_PATH + "/final_model")
else:
    model = AutoModelForTokenClassification.from_pretrained(MODEL_SAVE_PATH + "/final_model")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_PATH + "/final_model")

    trainer = Trainer(
        model=model,
        args=training_args,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.090564,0.915631,0.929653,0.922589
2,0.273386,0.082970,0.942014,0.945978,0.943992
3,0.055896,0.078649,0.944519,0.951195,0.947845
4,0.031053,0.077281,0.946417,0.951195,0.948800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# EVALUATION

In [57]:
predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=2)
labels = predictions.label_ids

true_preds = []
true_labels = []

for pred, lab in zip(preds, labels):
    cur_preds = []
    cur_labels = []

    for p_i, l_i in zip(pred, lab):
        if l_i != -100:
            cur_preds.append(id2label[p_i])
            cur_labels.append(id2label[l_i])

    true_preds.append(cur_preds)
    true_labels.append(cur_labels)
    
print(classification_report(true_labels, true_preds))

              precision    recall  f1-score   support

         LOC       0.93      0.93      0.93      1668
        MISC       0.78      0.81      0.80       702
         ORG       0.87      0.91      0.89      1661
         PER       0.97      0.96      0.96      1617

   micro avg       0.90      0.92      0.91      5648
   macro avg       0.89      0.90      0.89      5648
weighted avg       0.90      0.92      0.91      5648

